# Latih RF-DETR untuk deteksi kerusakan jalan

Notebook ini melatih model **RF-DETR Nano** (lisensi Apache 2.0) untuk mendeteksi kerusakan jalan, lalu mengekspornya ke **ONNX** supaya bisa dipakai offline di aplikasi *Penilai Kondisi Jalan*.

**Sebelum mulai**
1. *Runtime → Change runtime type → T4 GPU*.
2. Buat akun gratis di [Roboflow](https://roboflow.com), lalu salin API key dari *Settings → API Keys*.
3. Di Colab, buka panel *Secrets* (ikon kunci di kiri), tambahkan `ROBOFLOW_API_KEY`, dan aktifkan *Notebook access*. Jangan tempel kunci langsung di sel kode.

**Hasil akhir:** `rfdetr_jalan.onnx` dan `classes.json`. Letakkan keduanya di folder `models/` pada proyek aplikasi.

## 1. Cek GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Pasang library

In [ ]:
!pip install -q "rfdetr[train,onnx]" roboflow onnxruntime supervision

## 3. Pengaturan

Pilih dataset kerusakan jalan di [Roboflow Universe](https://universe.roboflow.com/search?q=rdd2022). Dari URL `universe.roboflow.com/<WORKSPACE>/<PROJECT>`, isi `WORKSPACE` dan `PROJECT`. Nomor `VERSION` ada di halaman *Dataset* proyek itu.

Kandidat (periksa daftar kelas dan lisensinya di halaman proyek):
- [`nzy/rdd-m2mwd`](https://universe.roboflow.com/nzy/rdd-m2mwd): kelas D00, D10, D20, D40
- [`sadekaly/rdd2022-rbs8t`](https://universe.roboflow.com/sadekaly/rdd2022-rbs8t): sekitar 7.295 gambar
- [`floweryolo/rdd-pqjaa`](https://universe.roboflow.com/floweryolo/rdd-pqjaa): D00, D10, D20, D40, Repair

Kode kelas RDD: **D00** retak memanjang · **D10** retak melintang · **D20** retak buaya · **D40** lubang.

In [ ]:
WORKSPACE = "nzy"
PROJECT = "rdd-m2mwd"
VERSION = 1             # ganti sesuai nomor versi di halaman dataset

EPOCHS = 50             # mulai 50; naikkan bila mAP masih naik di akhir pelatihan
SIMPAN_KE_DRIVE = True  # simpan checkpoint ke Google Drive agar tidak hilang bila Colab terputus

## 4. Unduh dataset (format COCO)

In [ ]:
import os
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("coco")
DATASET_DIR = dataset.location
print(DATASET_DIR, os.listdir(DATASET_DIR))

## 5. Periksa kelas dan jumlah label

Pastikan kelasnya memang jenis kerusakan jalan dan ada folder `train` serta `valid`.

In [ ]:
import json
from collections import Counter

for split in ["train", "valid", "test"]:
    path = os.path.join(DATASET_DIR, split, "_annotations.coco.json")
    if not os.path.exists(path):
        print(f"{split}: tidak ada")
        continue
    coco = json.load(open(path))
    nama = {c["id"]: c["name"] for c in coco["categories"]}
    hitung = Counter(nama[a["category_id"]] for a in coco["annotations"])
    print(f"{split}: {len(coco['images'])} gambar, label per kelas: {dict(hitung)}")
print("Kategori:", nama)

## 6. Latih model

Dengan GPU T4, 50 epoch bisa memakan beberapa jam tergantung ukuran dataset.

In [ ]:
OUTPUT_DIR = "/content/output_rfdetr"
if SIMPAN_KE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/rfdetr_jalan"
os.makedirs(OUTPUT_DIR, exist_ok=True)

from rfdetr import RFDETRNano

model = RFDETRNano()
model.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size="auto",
    lr=1e-4,
    output_dir=OUTPUT_DIR,
)

## 7. Coba model pada beberapa gambar validasi

In [ ]:
import glob
import supervision as sv
from PIL import Image

CKPT = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
model = RFDETRNano(pretrain_weights=CKPT)

contoh = sorted(glob.glob(os.path.join(DATASET_DIR, "valid", "*.jpg")))[:4]
for path in contoh:
    img = Image.open(path).convert("RGB")
    det = model.predict(img, threshold=0.4)
    label = [f"{n} {c:.2f}" for n, c in zip(det.data["class_name"], det.confidence)]
    hasil = sv.BoxAnnotator().annotate(img.copy(), det)
    hasil = sv.LabelAnnotator().annotate(hasil, det, label)
    display(hasil.resize((512, int(512 * hasil.height / hasil.width))))

## 8. Ekspor ke ONNX

In [ ]:
import shutil

os.chdir("/content")
model = RFDETRNano(pretrain_weights=CKPT)
model.export()  # format ONNX, disimpan di folder output/

onnx_asal = sorted(glob.glob("/content/output/*.onnx"), key=os.path.getmtime)[-1]
shutil.copy(onnx_asal, "/content/rfdetr_jalan.onnx")
print("ONNX:", onnx_asal)

## 9. Cocokkan hasil ONNX dengan model asli, lalu simpan daftar kelas

Sel ini memakai dekoder ONNX yang **sama** dengan yang akan dipakai aplikasi, lalu membandingkannya dengan prediksi model PyTorch. Daftar nama kelas diambil dari model itu sendiri, supaya nomor kelas di ONNX pasti cocok.

In [ ]:
import numpy as np
import onnxruntime as ort

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


def deteksi_onnx(sess, img, ambang=0.4):
    h, w = sess.get_inputs()[0].shape[2:4]
    x = np.asarray(img.convert("RGB").resize((w, h), Image.BILINEAR), np.float32) / 255.0
    x = ((x - MEAN) / STD).transpose(2, 0, 1)[None]
    nama_keluar = [o.name for o in sess.get_outputs()]
    keluar = dict(zip(nama_keluar, sess.run(None, {sess.get_inputs()[0].name: x})))
    kotak, logit = keluar["dets"][0], keluar["labels"][0]
    prob = (1 / (1 + np.exp(-np.clip(logit, -88, 88))))[:, :-1]  # slot terakhir = latar
    kelas, skor = prob.argmax(1), prob.max(1)
    pilih = skor > ambang
    cx, cy, bw, bh = kotak[pilih].T
    xyxy = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1)  # ternormalisasi 0-1
    return xyxy, kelas[pilih], skor[pilih]


sess = ort.InferenceSession("/content/rfdetr_jalan.onnx", providers=["CPUExecutionProvider"])
peta_kelas, cocok, total = {}, 0, 0
for path in sorted(glob.glob(os.path.join(DATASET_DIR, "valid", "*.jpg")))[:30]:
    img = Image.open(path).convert("RGB")
    det = model.predict(img, threshold=0.4)
    for cid, nama in zip(det.class_id, det.data["class_name"]):
        peta_kelas[int(cid)] = str(nama)
    _, kelas_onnx, _ = deteksi_onnx(sess, img)
    total += 1
    cocok += sorted(kelas_onnx.tolist()) == sorted(int(c) for c in det.class_id)

print(f"Kelas ONNX sama dengan PyTorch pada {cocok}/{total} gambar")
print("Peta kelas:", dict(sorted(peta_kelas.items())))
assert cocok >= 0.8 * total, "Dekoder ONNX tidak cocok dengan model asli. Jangan pakai hasil ekspor ini dulu."

nama_keluar = [o.name for o in sess.get_outputs()]
jumlah_kelas = sess.get_outputs()[nama_keluar.index("labels")].shape[-1] - 1
json.dump({"kelas": {str(k): v for k, v in sorted(peta_kelas.items())}, "jumlah_kelas": jumlah_kelas,
           "model": "RF-DETR Nano", "dataset": f"{WORKSPACE}/{PROJECT}/{VERSION}"},
          open("/content/classes.json", "w"), indent=1)
print(open("/content/classes.json").read())

## 10. Ukur kecepatan di CPU

Perkiraan kasar; CPU Colab berbeda dengan laptop.

In [ ]:
import time

img = Image.open(contoh[0]).convert("RGB")
deteksi_onnx(sess, img)
t = time.perf_counter()
for _ in range(10):
    deteksi_onnx(sess, img)
print(f"{(time.perf_counter() - t) / 10 * 1000:.0f} ms per gambar (CPU Colab)")

## 11. Unduh hasil

Letakkan `rfdetr_jalan.onnx` dan `classes.json` di folder `models/` proyek aplikasi.

In [ ]:
from google.colab import files

if SIMPAN_KE_DRIVE:
    shutil.copy("/content/rfdetr_jalan.onnx", OUTPUT_DIR)
    shutil.copy("/content/classes.json", OUTPUT_DIR)
files.download("/content/rfdetr_jalan.onnx")
files.download("/content/classes.json")